# MTH3302 : Méthodes probabilistes et statistiques pour l'I.A.

Anne Martin<br/>
Polytechnique Montréal<br/>

---
# Projet H2026 : Prédiction de la quantité de particules fines dans l'air de Montréal

**But** : Prédire l'IQA des particules fines (PM) dans l'air de Montréal pour l'année 2025.

**Contexte** : La Ville de Montréal mesure la qualité de l'air via l'indice IQA (valeur 50 = limite acceptable). Les particules fines (PM) en suspension peuvent présenter des risques sanitaires. Données disponibles pour 7 stations : Saint-Joseph, Saint-Jean-Baptiste, Aéroport de Montréal, Caserne 17, York/Roberval, Saint-Dominique et Anjou.

### Données

**Fichiers fournis** (à déposer dans le répertoire du calepin) :
- qualite-de-lair_train.csv : IQA des 7 stations (2007-2024)
- meteo_train.csv : observations météo de l'aéroport de Montréal (2007-2024)
- qualite-de-lair_test.csv : données 2025 (sans PM)
- meteo_test.csv : données météo 2025

**Variables qualité de l'air** : stationId, nom, latitude, longitude, Élévation, Date, PM, O3, NO2, SO2

**Variables météo** : latitude, longitude, Date, pluie, neige, precip_tot, neige_au_sol, hum_rel (max/min/moy), pt_rosee, vitesse_vent (max/min/moy), visibilite (max/min/moy), pression (max/min/moy), temp (max/min/moy)

**Objectif** : Prédire 2555 valeurs (7 stations × 365 jours). Évaluation par RMSE sur Kaggle.

### Consignes

- Équipe de 3 à 5 personnes
- Au moins une soumission sur Kaggle
- Un calepin .ipynb par équipe (rapport final)
- Langage Julia obligatoire
- Démarche rigoureusement justifiée
- Maximum 2 soumissions/jour/équipe sur Kaggle

### Conseils

- **Reproductibilité** : le calepin doit permettre à une personne extérieure de comprendre et reproduire les résultats
- **Analyse exploratoire** : essentielle pour développer l'expertise et identifier des pistes d'amélioration
- **Données manquantes** : développer une méthode d'imputation appropriée
- **Données aberrantes** : peuvent dérailler le modèle si non traitées
- **Documentation** : mentionner les essais infructueux avec leurs raisons possibles d'échec
- **Validation locale** : bien tester avant de soumettre (limite de 2 soumissions/jour)

In [ ]:
using CSV
using DataFrames
using Dates
using GLM
using Statistics
using StatsBase
using CategoricalArrays
using Plots
using DataFrames, CSV, Dates, GLM, Statistics, StatsBase, CategoricalArrays
using Gadfly


---
## 1. Chargement de données

In [ ]:
# données de la qualité de l'air
train_air = CSV.read("data/qualite-de-lair_train.csv", DataFrame)
first(train_air, 5)

In [ ]:
# données météo de la station de l'aéroport de Montréal
train_meteo = CSV.read("data/meteo_train.csv", DataFrame)
first(train_meteo, 5)

In [ ]:
train = outerjoin(train_air, train_meteo[:,3:end], on=:Date)
first(train, 5)

### 1.1 Creation du jeu de validation temporel

Pour respecter la nature temporelle du problème, nous séparons les données chronologiquement plutôt qu'aléatoirement. L'année 2024 sert d'ensemble de validation, les années 2007-2023 servent à l'entraînement.

In [ ]:
train = sort(train, :Date)

train_temporal = filter(row -> year(row.Date) < 2024, train)
validation_temporal = filter(row -> year(row.Date) == 2024, train)
validation_temporal.mois = month.(validation_temporal.Date)
println("Periode train_temporal : ", minimum(train_temporal.Date), " a ", maximum(train_temporal.Date))
println("Periode validation_temporal : ", minimum(validation_temporal.Date), " a ", maximum(validation_temporal.Date))
println("Taille train_temporal : ", size(train_temporal))
println("Taille validation_temporal : ", size(validation_temporal))

train=train_temporal

first(validation_temporal, 5)

---
## 2. Analyse exploratoire

Analyse exploratoire initiale pour se familiariser avec les données.

### 2.1 Nettoyage et sanity check des donnees

Vérification de la cohérence des données avant modélisation.

**Points vérifiés** : dates complètes, doublons, couverture temporelle, variables incomplètes, anomalies.

In [ ]:
train = sort(train, :Date)

function pct_missing(x)
    100 * count(ismissing, x) / length(x)
end

function iqr_outlier_count(x)
    valeurs = collect(skipmissing(x))
    valeurs_num = [v for v in valeurs if v isa Real]
    if length(valeurs_num) < 4
        return 0
    end
    q1 = quantile(valeurs_num, 0.25)
    q3 = quantile(valeurs_num, 0.75)
    iqr = q3 - q1
    borne_inf = q1 - 1.5 * iqr
    borne_sup = q3 + 1.5 * iqr
    return count(v -> v < borne_inf || v > borne_sup, valeurs_num)
end

function detect_problem(x)
    miss = pct_missing(x)
    valeurs = collect(skipmissing(x))
    valeurs_num = [v for v in valeurs if v isa Real]

    if miss == 100
        return "variable entierement manquante"
    elseif miss > 20
        return "forte proportion de valeurs manquantes"
    elseif miss > 0
        return "valeurs manquantes a traiter"
    elseif !isempty(valeurs_num) && length(unique(valeurs_num)) <= 1
        return "variable quasi constante"
    elseif !isempty(valeurs_num) && iqr_outlier_count(x) > 0
        return "valeurs aberrantes potentielles (regle IQR)"
    else
        return "aucun probleme majeur detecte"
    end
end

function propose_action(problem)
    if problem == "variable entierement manquante"
        return "exclure la variable"
    elseif problem == "forte proportion de valeurs manquantes"
        return "evaluer si la variable doit etre exclue ou imputee prudemment"
    elseif problem == "valeurs manquantes a traiter"
        return "imputer ou retirer selon le contexte"
    elseif problem == "variable quasi constante"
        return "verifier son utilite predictive"
    elseif problem == "valeurs aberrantes potentielles (regle IQR)"
        return "inspecter visuellement puis corriger, tronquer ou conserver avec justification"
    else
        return "conserver telle quelle"
    end
end

dates_uniques = sort(unique(skipmissing(train.Date)))
date_min = minimum(dates_uniques)
date_max = maximum(dates_uniques)
dates_attendues = collect(date_min:Day(1):date_max)
dates_manquantes = setdiff(dates_attendues, dates_uniques)
ordre_chronologique = issorted(train.Date)
nb_doublons_station_date = nrow(train) - nrow(unique(select(train, [:stationId, :Date])))

resume_stations = combine(
    groupby(dropmissing(select(train, [:stationId, :nom, :Date])), [:stationId, :nom]),
    :Date => minimum => :date_min,
    :Date => maximum => :date_max,
    nrow => :nb_lignes,

resume_stations.complet =
    (resume_stations.date_min .== date_min) .&
    (resume_stations.date_max .== date_max) .&
    (resume_stations.nb_lignes .== length(dates_attendues))

sort!(resume_stations, :stationId)

sanity_global = DataFrame(
    verification = [
        "ordre chronologique",
        "dates manquantes",
        "doublons (stationId, Date)",
        "couverture complete des stations",
    ],
    resultat = [
        string(ordre_chronologique),
        string(length(dates_manquantes)),
        string(nb_doublons_station_date),
        string(all(resume_stations.complet)),
    ],
    interpretation = [
        ordre_chronologique ? "aucun probleme detecte" : "ordre a corriger avant analyse temporelle",
        isempty(dates_manquantes) ? "aucune date manquante sur la periode observee" : "verifier les trous temporels",
        nb_doublons_station_date == 0 ? "aucun doublon detecte" : "des doublons doivent etre inspectes",
        all(resume_stations.complet) ? "toutes les stations couvrent la periode complete" : "certaines stations sont incompletes",
    ],

sanity_table = DataFrame(
    variable = String[],
    pct_missing = Float64[],
    type = String[],
    probleme_detecte = String[],
    action_proposee = String[],

for nom_col in names(train)
    col = train[!, nom_col]
    probleme = detect_problem(col)
    push!(sanity_table, (
        string(nom_col),
        round(pct_missing(col), digits = 2),
        string(eltype(col)),
        probleme,
        propose_action(probleme),
    ))
end

sort!(sanity_table, :pct_missing, rev = true)

println("Resume global du sanity check")
display(sanity_global)

println("\nCouverture temporelle par station")
display(resume_stations)

println("\nApercu des dates manquantes (s'il y en a)")
display(first(dates_manquantes, min(10, length(dates_manquantes))))

println("\nTableau de synthese par variable")
sanity_table

### 2.1.1 Valeurs manquantes de `neige_au_sol` et saisonnalite

**Hypothèse** : Les valeurs manquantes de `neige_au_sol` suivent une logique saisonnière (absence de neige hors hiver).

**Test** : % de missing par saison météorologique.

In [ ]:
train.saison = map(train.Date) do d
    m = month(d)
    if m in [12, 1, 2]
        "hiver"
    elseif m in [3, 4, 5]
        "printemps"
    elseif m in [6, 7, 8]
        "ete"
    else
        "automne"
    end
end

ordre_saisons = ["hiver", "printemps", "ete", "automne"]

missing_neige_par_saison = combine(groupby(train, :saison),
    :neige_au_sol => (x -> count(ismissing, x)) => :nb_missing,
    nrow => :nb_total,

missing_neige_par_saison.pct_missing =
    round.(100 .* missing_neige_par_saison.nb_missing ./ missing_neige_par_saison.nb_total, digits = 2)

ordre_map = Dict(s => i for (i, s) in enumerate(ordre_saisons))
missing_neige_par_saison.ordre_saison = [ordre_map[s] for s in missing_neige_par_saison.saison]
sort!(missing_neige_par_saison, :ordre_saison)

display(missing_neige_par_saison[:, [:saison, :nb_missing, :nb_total, :pct_missing]])

plot_missing_neige_saison = Gadfly.plot(
    missing_neige_par_saison,
    x = :saison,
    y = :pct_missing,
    Geom.bar,
    Guide.xlabel("Saison"),
    Guide.ylabel("% de valeurs manquantes"),
    Guide.title("Pourcentage de valeurs manquantes de neige_au_sol par saison"),

plot_missing_neige_saison

In [ ]:
# Cette cellule a été supprimée car l'imputation de neige_au_sol 
# est effectuée plus loin dans la section 2.1.3
# (voir cellule d'imputation des variables météo)

println("Imputation de neige_au_sol effectuée dans la section 2.1.3")

**Décision**

Missing structurés par saison (rares en hiver, fréquents en été/automne) → absence de neige plutôt que perte d'information.

**Action** : Imputation à 0 pour le modèle de base.

### 2.1.2 Valeurs manquantes de `SO2` : structure par station, dans le temps et lien avec `PM`

**Hypothèse** : Les valeurs manquantes de `SO2` dépendent de la station ou du temps.

**Tests** : % de missing par station, évolution temporelle, corrélation SO2-PM.

**Objectif** : Décider de conserver/transformer/exclure la variable.

In [ ]:
### Test 1 : missing SO2 par station

missing_so2_par_station = combine(groupby(train, [:stationId, :nom]),
    :SO2 => (x -> count(ismissing, x)) => :nb_missing,
    nrow => :nb_total,

missing_so2_par_station.pct_missing =
    round.(100 .* missing_so2_par_station.nb_missing ./ missing_so2_par_station.nb_total, digits = 2)

sort!(missing_so2_par_station, :pct_missing, rev = true)

display(missing_so2_par_station)

In [ ]:
### Test 2 : missing SO2 dans le temps (par annee)

train.annee = year.(train.Date)

missing_so2_par_annee = combine(groupby(train, :annee),
    :SO2 => (x -> count(ismissing, x)) => :nb_missing,
    nrow => :nb_total,

missing_so2_par_annee.pct_missing =
    round.(100 .* missing_so2_par_annee.nb_missing ./ missing_so2_par_annee.nb_total, digits = 2)

sort!(missing_so2_par_annee, :annee)

display(missing_so2_par_annee)

plot_missing_so2_temps = Gadfly.plot(
    missing_so2_par_annee,
    x = :annee,
    y = :pct_missing,
    Geom.bar,
    Guide.xlabel("Annee"),
    Guide.ylabel("% de valeurs manquantes"),
    Guide.title("Pourcentage de valeurs manquantes de SO2 par annee"),

plot_missing_so2_temps

In [ ]:
### Test 3 : correlation SO2 vs PM sur les donnees non manquantes

so2_pm_obs = dropmissing(select(train, [:SO2, :PM]))

corr_so2_pm = cor(so2_pm_obs.SO2, so2_pm_obs.PM)

println("Correlation SO2 vs PM (donnees non manquantes) : ", round(corr_so2_pm, digits = 4))

**Décision**

3/7 stations ont 100% de missing pour `SO2` (absence de capteur). Corrélation SO2-PM très faible (≈ 0.06).

**Action** : `SO2` exclue du modèle de base (via `:box_base`) — faible signal prédictif, mesure non homogène.

In [ ]:
# =========================
# Suppression de SO2 et neige
# =========================

println("Suppression de SO2 et neige du DataFrame...")

# Supprimer SO2 et neige
select!(train, Not([:SO2, :neige]))

println("✓ SO2 et neige supprimés")
println("Colonnes restantes : ", ncol(train))

### 2.1.3 Strategie d'imputation des valeurs manquantes

Imputation spécifique par type de variable pour préserver le signal utile.

**Principes** :
- Variables dynamiques (polluants) : forward fill (préserver inertie temporelle)
- Variables météo : interprétation physique
- `neige`, `neige_au_sol` : imputation à 0 (absence de neige hors saison)

In [ ]:
# =========================
# Imputation variables dynamiques (polluants)
# =========================

function forward_fill!(df, col)
    last_val = missing
    for i in 1:nrow(df)
        if !ismissing(df[i, col])
            last_val = df[i, col]
        else
            df[i, col] = last_val
        end
    end
end

function backward_fill!(df, col)
    next_val = missing
    for i in nrow(df):-1:1
        if !ismissing(df[i, col])
            next_val = df[i, col]
        else
            df[i, col] = next_val
        end
    end
end

function get_season(d::Date)
    m = month(d)
    if m in (12, 1, 2)
        return "Hiver"
    elseif m in (3, 4, 5)
        return "Printemps"
    elseif m in (6, 7, 8)
        return "Été"
    else
        return "Automne"
    end
end

# imputation features seulement
for col in [:NO2, :O3]
    for g in groupby(train, :stationId)
        sort!(g, :Date)
        forward_fill!(g, col)
        backward_fill!(g, col)
    end
end

# ne pas imputer la cible
dropmissing!(train, :PM)


println("✓ Imputation des polluants (PM, NO2, O3) par forward/backward fill")

# =========================
# Imputation variables météo
# =========================

train.saison = get_season.(train.Date)

sort!(train, [:stationId, :Date])

for g in groupby(train, :stationId)
    neige = g.neige_au_sol

    # interpolation simple forward fill + backward fill
    for i in 2:length(neige)
        if ismissing(neige[i]) && !ismissing(neige[i-1])
            neige[i] = neige[i-1]
        end
    end

    for i in length(neige)-1:-1:1
        if ismissing(neige[i]) && !ismissing(neige[i+1])
            neige[i] = neige[i+1]
        end
    end
end
# neige_au_sol : imputation à 0 (absence de neige hors saison)
train[!, :neige_au_sol] = coalesce.(train[!, :neige_au_sol], 0)

println("✓ Imputation de neige_au_sol en fonction de la saison")

println("\nImputation terminée")

### 2.1.4 Verification des valeurs manquantes de `precip_tot` et `pluie` dans le temps

**Hypothèse** : Si les missing sont faibles et sans structure temporelle, l'imputation à 0 est justifiée.

**Tests** : % de missing par mois, comparaison visuelle des distributions temporelles.

In [ ]:
train.mois = month.(train.Date)

function distribution_missing_par_mois(df, col)
    res = combine(groupby(df, :mois),
        col => (x -> count(ismissing, x)) => :nb_missing,
        nrow => :nb_total,

    res.pct_missing = round.(100 .* res.nb_missing ./ res.nb_total, digits = 2)
    res.variable .= string(col)

    return res
end

missing_precip = distribution_missing_par_mois(train, :precip_tot)
missing_pluie  = distribution_missing_par_mois(train, :pluie)

# fusion
distribution_missing = vcat(missing_precip, missing_pluie)

sort!(distribution_missing, [:mois, :variable])

# graphe combiné
plot_missing_combined = Gadfly.plot(
    distribution_missing,
    x = :mois,
    y = :pct_missing,
    color = :variable,
    Geom.bar(position = :dodge),
    Guide.xlabel("Mois"),
    Guide.ylabel("% valeurs manquantes"),
    Guide.title("Comparaison du missing : precip_tot vs pluie"),

plot_missing_combined

**Décision**

`pluie` et `precip_tot` ont des distributions de missing similaires (redondance partielle). `precip_tot` est plus globale.

**Action** : Conservation de `pluie`.

In [ ]:
train[!, :pluie] = coalesce.(train[!, :pluie], 0)
select!(train, Not(:precip_tot))

### 2.1.5 Traitement des doublons station-date

1387 doublons (stationId, Date) détectés.

**Stratégie** : Conservation de la première occurrence chronologique.

**Impact** : Réduction de ~1387 lignes, garantie d'unicité par paire (station, date).

In [ ]:
# Vérifier le nombre de doublons avant traitement
nb_avant = nrow(train)
nb_doublons_avant = nb_avant - nrow(unique(select(train, [:stationId, :Date])))

println("Avant traitement :")
println("  - Nombre total de lignes : ", nb_avant)
println("  - Nombre de doublons : ", nb_doublons_avant)

# Supprimer les doublons en gardant la première occurrence
train = unique(train, [:stationId, :Date])

# Vérifier après traitement
nb_apres = nrow(train)
nb_doublons_apres = nb_apres - nrow(unique(select(train, [:stationId, :Date])))

println("\nAprès traitement :")
println("  - Nombre total de lignes : ", nb_apres)
println("  - Nombre de doublons : ", nb_doublons_apres)
println("  - Lignes supprimées : ", nb_avant - nb_apres)

# S'assurer que le DataFrame est toujours trié par date
sort!(train, :Date)

### 2.1.6 Traitement des valeurs aberrantes

**Stratégie** :
- Remplacement par `missing` des valeurs physiquement impossibles uniquement
- Conservation des valeurs extrêmes mais plausibles
- Pas de suppression automatique basée sur l'IQR (préserve les événements environnementaux réels)

In [ ]:
# Variables detectees comme ayant des outliers potentiels
vars_aberrantes = filter(
    :probleme_detecte => ==("valeurs aberrantes potentielles (regle IQR)"),
    sanity_table

display(vars_aberrantes)

# Fonction : retourne les bornes physiques selon le nom de colonne
function get_bornes_physiques(col::Symbol)
    nom = String(col)

    if startswith(nom, "hum_rel")
        return (0.0, 100.0)

    elseif startswith(nom, "vitesse_vent")
        return (0.0, Inf)

    elseif startswith(nom, "visibilite")
        return (0.0, Inf)

    elseif startswith(nom, "pression")
        return (80.0, 110.0)   # kPa, plus plausible que 800-1100

    elseif startswith(nom, "temp")
        return (-50.0, 50.0)

    elseif nom in ["PM", "NO2", "O3", "pluie", "precip_tot", "neige"]
        return (0.0, Inf)

    else
        return nothing
    end
end

# Tableau de suivi des corrections
resume_corrections = DataFrame(
    variable = String[],
    borne_inf = Float64[],
    borne_sup = Float64[],
    nb_corrigees = Int[]

# Correction
for row in eachrow(vars_aberrantes)
    col = Symbol(row.variable)
    bornes = get_bornes_physiques(col)

    if isnothing(bornes)
        continue
    end

    borne_inf, borne_sup = bornes
    nb_corrigees = 0

    train[!, col] = map(train[!, col]) do x
        if ismissing(x)
            missing
        elseif x < borne_inf || x > borne_sup
            global nb_corrigees += 1
            missing
        else
            x
        end
    end

    push!(resume_corrections, (
        String(col),
        borne_inf,
        isfinite(borne_sup) ? borne_sup : Inf,
        nb_corrigees
    ))
end

display(resume_corrections)

In [ ]:
println("===== SANITY CHECK FINAL =====")

# =========================
# 0. Nettoyage colonnes inutiles
# =========================

println("\nVariables supprimées du DataFrame :")
println("  - SO2 : supprimée")
println("  - neige : supprimée")
println("  - precip_tot : supprimée")

# =========================
# 1. Missing restant
# =========================

println("\nMissing restants :")
for col in names(train)
    nb = count(ismissing, train[!, col])
    if nb > 0
        println(col, " : ", nb)
    end
end

# =========================
# 2. Vérification bornes physiques
# =========================

println("\nValeurs physiques invalides :")

checks = [
    (:hum_rel_max, x -> x > 100 || x < 0),
    (:hum_rel_min, x -> x > 100 || x < 0),
    (:pluie, x -> x < 0),
    (:vitesse_vent_moy, x -> x < 0)
]

for (col, cond) in checks
    if col in names(train)
        nb = count(x -> !ismissing(x) && cond(x), train[!, col])
        if nb > 0
            println(col, " valeurs invalides : ", nb)
        end
    end
end

# =========================
# 3. Vérification doublons station-date
# =========================

nb_doublons = nrow(train) - nrow(unique(select(train, [:stationId, :Date])))
println("\nDoublons station-date : ", nb_doublons)

# =========================
# 4. Vérification tri temporel par station
# =========================

println("\nVerification ordre temporel par station :")

for g in groupby(train, :stationId)
    if !issorted(g.Date)
        println("Station non triée : ", first(g.stationId))
    end
end

println("\nSanity check terminé")

**Décision**

Aucune valeur physiquement impossible identifiée. Conservation des valeurs extrêmes (événements environnementaux réels).

---
## 3. Analyse exploratoire approfondie et feature engineering

**Objectifs** :
- Analyser les corrélations entre variables
- Visualiser les distributions et patterns temporels
- Créer de nouvelles features (variables dérivées, interactions)
- Identifier les variables les plus prédictives

### Hypothèse : Relation non linéaire entre Pm et les features

**Test** : Traver scatter léger et moyenne ou médiane de PM par bins de X

**Critère** : Verifier la forme de la relation

In [ ]:
import Pkg; Pkg.add("StatsPlots")

In [ ]:
using DataFrames, Statistics, StatsPlots

function nonlinear_profile_plot(df, xcol, ycol; nbins=20)
    d = dropmissing(select(df, [xcol, ycol]))
    x = Float64.(d[!, xcol])
    y = Float64.(d[!, ycol])

    edges = quantile(x, range(0, 1, length=nbins+1))
    edges = unique(edges)

    if length(edges) < 3
        error("Pas assez de valeurs distinctes pour créer des bins sur $(xcol)")
    end

    bins = [clamp(searchsortedlast(edges, xi), 1, length(edges)-1) for xi in x]

    d.bin = bins
    d.x_val = x
    d.y_val = y

    g = combine(groupby(d, :bin),
        :x_val => median => :x_mid,
        :y_val => median => :y_med,
        :y_val => mean   => :y_mean,
        nrow => :n

    p = StatsPlots.plot(g.x_mid, g.y_med,
        marker=:circle, lw=2, label="médiane",
        xlabel=String(xcol), ylabel=String(ycol),
        title=String(xcol))

    StatsPlots.plot!(p, g.x_mid, g.y_mean,
        marker=:square, lw=2, label="moyenne")

    return p
end

vars = [:visibilite_moy, :NO2, :hum_rel_moy, :pression_moy]

plots = [nonlinear_profile_plot(train, v, :PM, nbins=20) for v in vars]

StatsPlots.plot(plots..., layout=(2,2), size=(1000,700))

In [ ]:
# Profils non linéaires pour vitesse_vent_moy et O3
vars = [:vitesse_vent_moy, :O3]

plots = [nonlinear_profile_plot(train, v, :PM, nbins=20) for v in vars]

StatsPlots.plot(plots..., layout=(1,2), size=(1000,400))

In [ ]:
vars = [:pluie, :pt_rosee, :vitesse_vent_max]

plots = [nonlinear_profile_plot(train, v, :PM, nbins=20) for v in vars]

StatsPlots.plot(plots..., layout=(1,3), size=(1200,400))

À retenir

NO2 : relation croissante quasi linéaire
utile, mais pas besoin de non-linéarité

visibilite_moy : relation décroissante non linéaire, avec plateau
candidate à transformation

hum_rel_moy : relation croissante non linéaire
candidate à transformation

vitesse_vent_moy : relation décroissante non linéaire forte
candidate forte

vitesse_vent_max : relation décroissante non linéaire forte
candidate forte

O3 : forme en U nette
candidate forte à terme quadratique

### Hypothèse : Distribution de PM

**Test** : Comparer la distribution de PM selon la station.

**Critère** : Différences nettes entre stations → `stationId` est une variable importante.

In [ ]:
println("===== Statistiques descriptives de PM =====")

describe(train.PM)

histogram(train.PM,
    bins=80,
    title="Histogramme de PM",
    xlabel="PM",
    ylabel="Fréquence",
    label=false)

In [ ]:

println("===== Statistiques descriptives de PM par station =====")

# Stats par station
stats_station = combine(groupby(train, :stationId),
    :PM => mean => :PM_mean,
    :PM => median => :PM_median,
    :PM => std => :PM_std,
    :PM => length => :n_obs

println(stats_station)

Plots.gr()

plots = []

for g in groupby(train, :stationId)
    p = Plots.histogram(g.PM,
        bins=40,
        title="Station $(first(g.stationId))",
        xlabel="PM",
        ylabel="Fréquence",
        label=false
    push!(plots, p)
end

Plots.plot(plots..., layout=(3,3), size=(1000,800))

In [ ]:
combine(groupby(train, :stationId),
    :PM => mean => :mean,
    :PM => median => :median,
    :PM => std => :std,
    :PM => maximum => :max,
    :PM => x -> quantile(x, 0.9) => :q90


### 3.2 Décision : Transformation de PM

Les distributions de PM sont globalement similaires entre stations, suggérant un effet marginal modéré.

Cependant, cela ne permet pas de conclure à l’inutilité de la variable `stationId`, qui peut capturer des effets locaux non visibles dans une analyse univariée.

### Hypothèse — Axe temporel

Le niveau de PM varie selon le temps (mois/année) en raison de facteurs saisonniers (température, chauffage, inversions thermiques).

**Test** : PM moyen par mois pour une année donnée.

**Critère** : Variation forte → variables temporelles (mois, année) importantes.

In [ ]:
# Filtrer 2023
train_2023 = filter(:annee => ==(2023), train)

# PM moyen par station et par mois
pm_station_month = combine(groupby(train_2023, [:nom, :mois]),
    :PM => mean => :PM_mean

sort!(pm_station_month, [:nom, :mois])

# Mois en abscisse
mois_vals = 1:12
mois_labels = monthabbr.(Date.(2023, mois_vals, 1))

# Une petite figure par station
stations = sort(unique(pm_station_month.nom))
plots_list = Plots.Plot[]

for stationId in stations
    df_station = filter(:nom => ==(stationId), pm_station_month)
    sort!(df_station, :mois)

    p = Plots.plot(
        df_station.mois, df_station.PM_mean,
        seriestype = :line,
        marker = :circle,
        xticks = (mois_vals, mois_labels),
        title = string(stationId),
        xlabel = "Mois",
        ylabel = "PM moyen",
        legend = false

    push!(plots_list, p)
end

Plots.plot(plots_list..., layout=(4, 2), size=(1100, 1200))

### Hypothèse — Volatilité temporelle par saison

La volatilité hebdomadaire de PM varie selon la saison et selon la station.

**Raison** : Certaines saisons/stations peuvent produire des variations plus brusques de PM.

**Test** : Calculer le PM moyen par station/saison/semaine, puis mesurer la volatilité.

### Hypothèse — Temporalité fine

Le PM varie à une échelle fine (semaine/jour) en raison des activités humaines et de la météo.

**Test** : PM moyen par jour de la semaine et par semaine de l'année.

**Critère** : Pattern hebdomadaire cohérent (ex: jours de semaine vs week-end).

In [ ]:


# 1. Fonction saison
function get_season(d::Date)
    m = month(d)
    if m in (12, 1, 2)
        return "Hiver"
    elseif m in (3, 4, 5)
        return "Printemps"
    elseif m in (6, 7, 8)
        return "Été"
    else
        return "Automne"
    end
end

# 2. Colonnes temporelles
train.saison_calc = get_season.(train.Date)
train.semaine_iso = week.(train.Date)

# Important : distinguer les semaines d'années différentes
train.annee_semaine = string.(train.annee, "-S", lpad.(string.(train.semaine_iso), 2, "0"))

# 3. PM moyen hebdomadaire par station et par saison
pm_weekly = combine(
    groupby(train, [:nom, :saison_calc, :annee_semaine]),
    :PM => mean => :PM_week_mean

# 4. Volatilité des semaines dans chaque saison/station
volatility_results = combine(
    groupby(pm_weekly, [:nom, :saison_calc]),
    :PM_week_mean => mean => :PM_mean_of_weeks,
    :PM_week_mean => std => :PM_week_std,
    :PM_week_mean => var => :PM_week_var,
    nrow => :n_weeks

sort!(volatility_results, [:saison_calc, :PM_week_std], rev=[false, true])

volatility_results

In [ ]:
saisons_order = ["Hiver", "Printemps", "Été", "Automne"]

plots_list = Plots.Plot[]

for saison in saisons_order
    df_s = filter(:saison_calc => ==(saison), volatility_results)
    sort!(df_s, :PM_week_std, rev=true)

    p = Plots.bar(
        df_s.nom,
        df_s.PM_week_std,
        title = "Volatilité hebdomadaire de PM — $saison",
        xlabel = "Station",
        ylabel = "Écart-type des moyennes hebdomadaires",
        legend = false,
        xrotation = 45

    push!(plots_list, p)
end

Plots.plot(plots_list..., layout=(2,2), size=(1200, 900))

In [ ]:
pm_weekly_station_season = combine(
    groupby(train, [:nom, :saison_calc, :annee, :semaine_iso]),
    :PM => mean => :PM_week_mean

plots_list2 = Plots.Plot[]

for saison in saisons_order
    df_s = filter(:saison_calc => ==(saison), pm_weekly_station_season)

    p = Plots.plot(
        title = "PM hebdomadaire — $saison",
        xlabel = "Semaine ISO",
        ylabel = "PM moyen hebdomadaire",
        legend = :topright

    for stationId in sort(unique(df_s.nom))
        df_station = filter(:nom => ==(stationId), df_s)
        sort!(df_station, [:annee, :semaine_iso])

        # index continu pour affichage
        x_vals = 1:nrow(df_station)

        Plots.plot!(p, x_vals, df_station.PM_week_mean,
            seriestype = :line,
            marker = :circle,
            label = stationId
    end

    push!(plots_list2, p)
end

Plots.plot(plots_list2..., layout=(2,2), size=(1200, 900))

### Conclusion — Volatilité temporelle

**Observation** : Volatilité de PM varie selon la saison (automne stable, hiver/été volatils). Certaines stations plus volatiles.

**Interprétation** : La saison influence non seulement le niveau moyen de PM, mais aussi sa variabilité.

**Décision** : Variables temporelles essentielles. Interaction station × saison à tester.

### Hypothèse — Vent

Une vitesse de vent élevée est associée à une diminution du PM (dispersion des particules).

**Test** : Corrélation entre vitesse du vent et PM, scatter plot.

**Critère** : Corrélation négative → variable vent importante.

In [ ]:
# Variables météo
vars_meteo = [:pluie, :neige_au_sol, :precip_tot,
              :hum_rel_max, :hum_rel_min, :hum_rel_moy,
              :temp_max, :temp_min, :temp_moy,
              :vitesse_vent_max, :vitesse_vent_min, :vitesse_vent_moy,
              :visibilite_moy,
              :pression_max, :pression_min, :pression_moy]

# Garder uniquement les colonnes présentes dans train
vars_present = filter(c -> c in names(train), vars_meteo)

# DataFrame final pour corrélation (avec la cible)
train_features_corr = select(train, [:PM; vars_present])
println("Nombre de lignes: ", nrow(train_features_corr))
println("Colonnes disponibles: ", names(train_features_corr))
println()
println("Corrélations PM vs Variables Météorologiques :")
println("=" ^ 60)

for var in vars_meteo
    if var in Symbol.(names(train_features_corr))
        # Calculer corrélation uniquement sur les paires non-missing
        df_temp = dropmissing(train_features_corr, [var, :PM])
        if nrow(df_temp) > 0
            c = cor(df_temp[!, var], df_temp.PM)
            println(rpad(string(var), 25), " => ", round(c, digits=4), " (", nrow(df_temp), " obs)")
        else
            println(rpad(string(var), 25), " => AUCUNE DONNÉE")
        end
    else
        println(rpad(string(var), 25), " => NON TROUVÉE DANS :box_base")
    end
end

In [ ]:
model_vent_multi = lm(@formula(PM ~ vitesse_vent_moy + vitesse_vent_max + vitesse_vent_min), train)

println(coeftable(model_vent_multi))

x = train.vitesse_vent_moy
y = train.PM

# prédictions
ŷ = GLM.predict(model_vent_multi)

Plots.scatter(x, y,
    markersize=2,
    alpha=0.3,
    label="Data",
    xlabel="Vent moyen",
    ylabel="PM"

Plots.plot!(x, ŷ,
    color=:red,
    label="Régression")

### Décision — Variables de vent

Variables de vent fortement corrélées entre elles (multicolinéarité). Coefficients instables et incohérents.

**Action** : Conservation uniquement de `vitesse_vent_moy` comme variable représentative. Suppression de `vitesse_vent_min` et `vitesse_vent_max`.

### Hypothèse — Température

La température influence PM, mais la relation peut ne pas être strictement linéaire (liée à la saison, dispersion/formation de particules).

**Test** : Scatter plot température vs PM, observer la forme de la relation.

In [ ]:
saisons = ["Hiver", "Printemps", "Été", "Automne"]
plots_list = Plots.Plot[]

for s in saisons
    df = filter(:saison_calc => ==(s), train)

    p = Plots.scatter(
        df.temp_moy, df.PM,
        markersize=2,
        alpha=0.3,
        xlabel="Température moyenne",
        ylabel="PM",
        title="Temp vs PM ($s)",
        label=false

    push!(plots_list, p)
end

Plots.plot(plots_list..., layout=(2,2), size=(1000,800))

In [ ]:
model1 = lm(@formula(PM ~ mois + stationId + vitesse_vent_moy), train)



In [ ]:
model2 = lm(@formula(PM ~ mois + stationId + vitesse_vent_moy + temp_moy), train)

### Décision — Température

Variable statistiquement significative, mais son ajout modifie fortement le coefficient du mois (redondance importante avec la saisonnalité).

**Action** : Température supprimée du modèle de base (contribution marginale insuffisante).

### Hypothèse — Humidité

L'humidité relative influence le niveau de PM (formation de particules fines, absorption d'eau, air stagnant).

**Direction attendue** : Humidité ↑ ⇒ PM ↑ (relation positive attendue).

In [ ]:
# corrélation par saisons
    
saisons = ["Hiver", "Printemps", "Été", "Automne"]

for s in saisons
    df = filter(:saison_calc => ==(s), train)

    corr = cor(df.hum_rel_moy, df.PM)

    println("Corrélation humidité vs PM ($s) : ", round(corr, digits=4))
end

In [ ]:
plots_list = Plots.Plot[]

for s in saisons
    df = filter(:saison_calc => ==(s), train)

    p = Plots.scatter(
        df.hum_rel_moy, df.PM,
        markersize=2,
        alpha=0.3,
        xlabel="Humidité relative",
        ylabel="PM",
        title="Humidité vs PM ($s)",
        label=false

    push!(plots_list, p)
end

Plots.plot(plots_list..., layout=(2,2), size=(1000,800))

In [ ]:
lm(@formula(PM ~ mois + stationId + vitesse_vent_moy + hum_rel_moy), train)

In [ ]:
lm(@formula(PM ~ mois + stationId + vitesse_vent_moy + hum_rel_moy*saison_calc), train)

### Conclusion — Humidité

Humidité significative, mais son effet dépend fortement de la saison (positif en hiver, négatif en été). Interaction non linéaire confirmée.

**Décision** : Interaction humidité × saison pertinente. Humidité seule insuffisante.

### Hypothèse — Visibilité

Une faible visibilité est associée à un niveau élevé de PM (particules en suspension réduisent la visibilité).

**Direction attendue** : Corrélation négative forte (visibilité ↓ ⇒ PM ↑).

In [ ]:
cor(train.visibilite_moy, train.PM)

In [ ]:
Plots.scatter(train.visibilite_moy, train.PM,
    xlabel="Visibilité",
    ylabel="PM",
    title="Visibilité vs PM",
    markersize=2,
    alpha=0.3,
    label=false)

In [ ]:
saisons = ["Hiver", "Printemps", "Été", "Automne"]

for s in saisons
    df = filter(:saison_calc => ==(s), train)
    println("Corr visibilité vs PM ($s): ",
        round(cor(df.visibilite_moy, df.PM), digits=4))
end

### Conclusion — Visibilité

Corrélation négative stable entre visibilité et PM. Effet plus marqué en été et automne. Relation non linéaire : faible visibilité permet des valeurs élevées de PM.

**Décision** : Variable potentiellement très prédictive.

### Hypothèse — Interaction météo × saison

Les effets des variables météo (vent, humidité) sur PM dépendent de la saison (conditions physiques différentes, mécanismes de pollution différents).

**Test** : Comparer modèle avec et sans interaction.

In [ ]:
lm(@formula(PM ~ mois + stationId + vitesse_vent_moy + hum_rel_moy), train)


In [ ]:
lm(@formula(PM ~ mois + stationId + vitesse_vent_moy*saison_calc + hum_rel_moy*saison_calc), train)

In [ ]:
lm(@formula(PM ~ mois + stationId + vitesse_vent_moy*mois + hum_rel_moy*mois), train)

### Conclusion EDA — PM

#### 1. Distribution cible
- PM fortement asymétrique (queue à droite)

→ transformation log potentiellement utile

---

#### 2. Spatialité
- différences faibles entre stations (moyenne similaire)
- variance différente selon stations

→ stationId conservé (effet secondaire)

---

#### 3. Temporalité (très forte)
- variation mensuelle marquée
- saisonnalité dominante

→ mois = variable clé

---

#### 4. Vent
- corrélation négative forte
- effet stable

→ variable principale conservée

---

#### 5. Température
- corrélation faible
- effet disparaît par saison

→ variable supprimée (redondante avec mois)

---

#### 6. Humidité
- corrélation modérée
- effet dépend de la saison

→ variable conservée
→ interaction nécessaire avec temps

---

#### 7. Visibilité
- corrélation négative forte
- relation non linéaire
- variable potentiellement dépendante de PM

→ mise de côté (risque de leakage)
→ disponible dans :box_base_plus pour tests

---

#### 8. Autres variables météo
- pression, pluie → faible signal univarié

→ mises de côté (disponibles dans :box_meteo_complete)


---

### 3.12 Le Système "Feature Box" (Moteur d'Inertie)

Cette cellule définit le moteur modulaire pour injecter à la volée nos options de Features.
Nous codons les fonctions de lissage et de décompte en `Julia pur` pour éviter les conflits de dépendance.
Chaque sous-module est activé par le dictionnaire `config`.

In [ ]:
using Statistics, Dates, DataFrames

# --- Fonctions utilitaires d'inertie temporelle --- #
function compute_rolling_mean(x::AbstractVector, window::Int)
    n = length(x)
    res = Vector{Union{Missing, Float64}}(undef, n)
    for i in 1:n
        start_idx = max(1, i - window + 1)
        sub = skipmissing(x[start_idx:i])
        res[i] = isempty(sub) ? missing : mean(sub)
    end
    return res
end

function compute_dry_days(precip::AbstractVector)
    n = length(precip)
    res = zeros(Int, n)
    counter = 0
    for i in 1:n
        p = precip[i]
        if ismissing(p) || p > 0.0
            counter = 0
        else
            counter += 1
        end
        res[i] = counter
    end
    return res
end

function compute_lag(x::AbstractVector, lag::Int)
    n = length(x)
    res = Vector{Union{Missing, eltype(x)}}(undef, n)
    for i in 1:n
        if i <= lag
            res[i] = missing
        else
            res[i] = x[i - lag]
        end
    end
    return res
end
function apply_feature_box!(df, config::Dict)
    println("Application de la Feature Box...")

    # 1. Cyclical Time Encoding
    if get(config, :add_cyclical, false)
        println(" -> Ajout des variables cycliques (sin/cos de l'année)")
        df.jour_annee     = dayofyear.(df.Date)
        df.jour_annee_sin = sin.(2π .* df.jour_annee ./ 365.25)
        df.jour_annee_cos = cos.(2π .* df.jour_annee ./ 365.25)
    end

    needs_temporal = get(config, :add_rolling, false) ||
                     get(config, :add_dry_days, false) ||
                     get(config, :add_lags, false)

    if needs_temporal
        println(" -> Tri temporel des stations en cours...")
        sort!(df, [:stationId, :Date])
        gdf = groupby(df, :stationId)

        if get(config, :add_rolling, false)
            println(" -> Ajout des moyennes glissantes")
            transform!(gdf,
                :vitesse_vent_moy => (x -> compute_rolling_mean(x, 3)) => :vitesse_vent_moy_3j,
                :vitesse_vent_moy => (x -> compute_rolling_mean(x, 7)) => :vitesse_vent_moy_7j,
                :temp_moy         => (x -> compute_rolling_mean(x, 3)) => :temp_moy_3j,
            )  # ✅ parenthèse fermante ajoutée
        end

        if get(config, :add_dry_days, false)
            println(" -> Ajout de l'accumulateur de sécheresse")
            precip_col = "precip_tot" in names(df) ? :precip_tot : :pluie
            transform!(gdf, precip_col => compute_dry_days => :jours_sans_pluie)
        end

        if get(config, :add_lags, false)
            println(" -> Ajout des décalages (Lags)")
            transform!(gdf,
                :NO2 => (x -> compute_lag(x, 1)) => :NO2_lag_1,
                :O3  => (x -> compute_lag(x, 1)) => :O3_lag_1,
            )  # ✅ parenthèse fermante ajoutée
        end
    end

    println("Feature Box appliquée avec succès !")
    return df
end

In [ ]:
println(names(train))


## Section 4 – Modélisation Baseline (Post-Nettoyage)

Cette section établit les bases propres du modèle prédictif avant d'entamer le feature engineering complexe (Phase 2).
- **Objectif** : Évaluer la capacité de généralisation.
- **Validation** : Hold-out strict sur des années complètes (2023 et 2024), car le but final est de prédire une année entière (2025) sans ré-entraînement.

In [ ]:
function annual_holdout_cv(df, formula)
    println("--- Validation Annuelle (Hold-Out) ---")
    
    years_to_test = [2022, 2024]
    response = formula.lhs.sym
    results = DataFrame(
        annee_test=Int[],
        rmse_train=Float64[],
        rmse_test=Float64[]
    
    for y in years_to_test
        df_train = filter(row -> year(row.Date) < y, df)
        df_test = filter(row -> year(row.Date) == y, df)
        
        if nrow(df_test) == 0
            continue
        end
        
        m = lm(formula, df_train)
        
        yhat_train = predict(m, df_train)
        yhat_test = predict(m, df_test)
        
        global_beta = β
        push!(results, (
            y,
            rmse(df_train[!, response], yhat_train),
            rmse(df_test[!, response], yhat_test)
        ))
    end
    
    return results
end

In [ ]:
# 1. Initialisation de model_df à partir de tes données nettoyées (train)
model_df = vcat(train_temporal, validation_temporal, cols=:union)
model_df.mois = month.(model_df.Date) # Uniformisation de la colonne mois

# 2. Configuration de la Feature Box (Phase 2)
# On active ici les variables d'inertie et cycliques
box_options = Dict(
    :add_cyclical  => true, # sin/cos du jour de l'année
    :add_rolling   => true, # moyennes glissantes 3j/7j
    :add_dry_days  => true, # jours sans pluie
    :add_lags      => true  # polluants J-1

# 3. Application du moteur de features
# Cette fonction trie par station/date et injecte les nouvelles colonnes
apply_feature_box!(model_df, box_options)

# 4. Nettoyage des Missing générés par les Lags/Rolling
# (Les 7 premiers jours de chaque station auront des missing pour les moyennes 7j)
dropmissing!(model_df)

println("model_df est prêt avec ", ncol(model_df), " colonnes.")


### 4.1 Modèle de Référence Minimal (Baseline)

Modèle linéaire standard utilisant les caractéristiques spatio-temporelles de base et les variables atmosphériques directes (sans cappings ni interactions complexes).

In [ ]:
# 1. Initialisation propre de model_df
model_df = vcat(train_temporal, validation_temporal, cols=:union)
model_df.mois = month.(model_df.Date) # Uniformisation de la colonne mois

# 2. Création des variables issues de l'EDA d'origine (Phase 1)
model_df.is_aero = model_df.stationId .== 66
o3_mean = mean(skipmissing(model_df.O3))
model_df.O3_centered_sq = (model_df.O3 .- o3_mean) .^ 2

using DataFrames, GLM, Statistics, Dates
rmse(y, yhat) = sqrt(mean(skipmissing((y .- yhat).^2)))

formula_base = @formula(PM ~ NO2 + O3 + stationId + mois + visibilite_moy + vitesse_vent_moy)
 
cv_base = annual_holdout_cv(model_df, formula_base)
println("RMSE Baseline :"); display(cv_base)

model_base = lm(formula_base, model_df)


### 4.2 Définition de l'indicateur de Stagnation

Comme observé dans l'EDA, un air stagnant (faible vent, faible visibilité) est un puissant propulseur de PM.

In [ ]:
wind_thresh = quantile(skipmissing(model_df.vitesse_vent_moy), 0.20)
vis_thresh  = quantile(skipmissing(model_df.visibilite_moy), 0.20)

model_df.low_wind = model_df.vitesse_vent_moy .<= wind_thresh
model_df.low_vis  = model_df.visibilite_moy .<= vis_thresh
model_df.stagnation = model_df.low_wind .& model_df.low_vis

model_df.inv_vitesse_vent_moy = 1.0 ./ (model_df.vitesse_vent_moy .+ 1e-6)
model_df.inv_visibilite_moy = 1.0 ./ (model_df.visibilite_moy .+ 1e-6)
model_df.hum_rel_moy_sq = model_df.hum_rel_moy .^ 2

# Mois catégoriel + indicateur juin pour souligner des effets locaux détectés 
using CategoricalArrays
model_df.mois = categorical(model_df.mois)
model_df.is_june = model_df.mois .== 6


### 4.3 Modèle Avancé (Promising Model issu de la Phase 1)

Ce modèle capture les effets de l'air stagnant et des transformations quadratiques. Il sert de point de départ robuste pour la Phase 2.

In [ ]:
formula_advanced = @formula(
    PM ~ NO2 + O3 + O3_centered_sq + stationId + mois  +
    inv_vitesse_vent_moy + inv_visibilite_moy + hum_rel_moy_sq +
    stagnation + NO2&stagnation + O3&stagnation + is_june

cv_advanced = annual_holdout_cv(model_df, formula_advanced)
println("RMSE Advance Model :"); display(cv_advanced)

model_advanced = lm(formula_advanced, model_df)


### 4.4 Évaluation de la Feature Box (Phase 2)

Nous allons activer notre boîte à outils temporelle, et ajouter ces nouvelles variables à notre modèle avancé pour voir si l'Inertie améliore nos capacités prédictives.

In [ ]:
# --- Application de la Feature Box --- #
box_options = Dict(
    :add_cyclical  => true,
    :add_rolling   => true,
    :add_dry_days  => true,
    :add_lags      => true

# On travaille sur une copie propre enrichie
model_df_enriched = apply_feature_box!(copy(model_df), box_options)
dropmissing!(model_df_enriched) # Les lags et rolling créent des missing en début de série

# --- Nouveau Modèle avec Inertie --- #
formula_inertia = @formula(
    PM ~ NO2 + O3 + O3_centered_sq + stationId + is_aero +
    inv_vitesse_vent_moy + inv_visibilite_moy + hum_rel_moy_sq +
    stagnation + NO2&stagnation + O3&stagnation +
    jour_annee_sin + jour_annee_cos + # Remplace 'mois' catégoriel
    vitesse_vent_moy_3j + vitesse_vent_moy_7j + temp_moy_3j +
    jours_sans_pluie + NO2_lag_1 + O3_lag_1

cv_inertia = annual_holdout_cv(model_df_enriched, formula_inertia)
println("\n--- RMSE Modèle avec Inertie Climatique ---")
display(cv_inertia)

model_inertia = lm(formula_inertia, model_df_enriched)


## Section 5 – Modèle Non-Linéaire Pénalisé (Régression Ridge)

Pour contourner l'interdiction des arbres de décision, nous allons créer des **centaines de variables croisées** (ex: `température * vent`, `jours_sans_pluie * stagnation`) pour forcer le modèle linéaire à "comprendre" des comportements spécifiques complexes.

Puisque cela créerait du sur-apprentissage massif avec les Moindres Carrés Ordinaires (OLS), nous utilisons la **Régression Ridge** vue en classe avec une pénalité L2 (`λ`) implémentée manuellement avec `LinearAlgebra`.

In [ ]:
using LinearAlgebra

# 1. L'Expansion Polynomiale Massive (Syntaxe ^2 pour croiser toutes les features inertielles)
# On retire PM de l'équation, StatsModels s'occupe de Matrix et Intercept
# 1. Création programmable de l'Expansion Massive (2-way interactions)
features_symb = [:NO2, :O3, :O3_centered_sq, :inv_vitesse_vent_moy, :inv_visibilite_moy, :hum_rel_moy_sq, :jour_annee_sin, :jour_annee_cos, :vitesse_vent_moy_3j, :temp_moy_3j, :jours_sans_pluie, :NO2_lag_1, :O3_lag_1]

base_terms = term.(features_symb)
interaction_list = Any[]
for i in 1:length(base_terms)
    push!(interaction_list, base_terms[i]) # Effets principaux
    for j in (i+1):length(base_terms)
        push!(interaction_list, base_terms[i] & base_terms[j]) # Croisements ^2
    end
end

formula_expansion = FormulaTerm(term(:PM), tuple(interaction_list...))

# On prépare le moteur (Schema) qui traduit la formule en matrice
f_schema = apply_schema(formula_expansion, schema(model_df_enriched))

# 2. La fonction Ridge Mathématique Manuelle (Vu en classe)
function fit_ridge(X::Matrix, y::Vector, λ::Float64)
    n = length(y)
    m = size(X, 2)
    
    # Matrice d'identité pour la pénalité (I est une UniformScaling de LinearAlgebra)
    # On crée une matrice car on ne veut généralement PAS pénaliser l'intercept (la 1ère colonne)
    L = λ * Matrix(I, m, m)
    L[1,1] = 0.0 # Ne pas tirer l'intercept vers 0
    
    β_ridge = (X'X + L) \ X'y
    
    return β_ridge
end

# 3. Validation croisée adaptée pour le Ridge
function ridge_annual_holdout_cv(df_enriched, f_schema, λ::Float64)
    years_to_test = [2022, 2024]
    
    results = DataFrame(annee_test=Int[], rmse_train=Float64[], rmse_test=Float64[])
    
    global_beta = Float64[]
    for y_test in years_to_test
        # Filtre
        df_train = filter(row -> year(row.Date) < y_test, df_enriched)
        df_test  = filter(row -> year(row.Date) == y_test, df_enriched)
        
        if nrow(df_test) == 0 continue end
        
        # Extraction des matrices X et vecteurs cibles y
        y_train, X_train = modelcols(f_schema, df_train)
        y_test_vec, X_test  = modelcols(f_schema, df_test)
        
        # Entraînement via les maths matricielles directes (Régression de Crête)
        # --- STANDARDISATION (Crucial pour Ridge) --- #
        μ = mean(X_train, dims=1)
        σ = std(X_train, dims=1)
        for j in 1:size(X_train,2)
            # Ne pas standardiser l'intercept (colonne 1) ni les constantes brutes
            if σ[1,j] == 0.0 || j == 1
                μ[1,j] = 0.0
                σ[1,j] = 1.0
            end
        end
        
        X_train_std = (X_train .- μ) ./ σ
        X_test_std  = (X_test .- μ) ./ σ
        
        # Entraînement sur données standardisées
        β = fit_ridge(X_train_std, y_train, λ)
        
        
        # Prédictions
        yhat_train = X_train_std * β
        yhat_test  = X_test_std * β
        
        rmse_train_val = sqrt(mean(skipmissing((y_train .- yhat_train).^2)))
        rmse_test_val  = sqrt(mean(skipmissing((y_test_vec .- yhat_test).^2)))
        
        global_beta = β
        push!(results, (y_test, rmse_train_val, rmse_test_val), promote=true)
    end
    
    return results, global_beta, std(skipmissing(df_enriched[df_enriched.Date .>= Date(2024,1,1), :PM]))
end

In [ ]:
println("Recherche du meilleur paramètre λ (Pénalité) sur la matrice à 91 dimensions...")

lambdas = [1e3, 1e4, 5e4, 1e5, 5e5, 1e6, 5e6, 1e7, 1e8]
best_lambda = 0.0
best_rmse = Inf

for lam in lambdas
    res, res_beta, std_pm = ridge_annual_holdout_cv(model_df_enriched, f_schema, Float64(lam))
    
    # On isole uniquement l'année 2024 pour notre prise de décision
    rmse_2024 = filter(r -> r.annee_test == 2024, res).rmse_test[1]
    
    println("λ = ", rpad(lam, 6), " | RMSE Test 2024 = ", round(rmse_2024, digits=4))
    
    # DEBUG: verify if β is completely empty or static
    println("    Debug β[1:5]: ", round.(res_beta[1:min(5, length(res_beta))], digits=4))
    println("    Debug std(PM_2024): ", std_pm)
    if rmse_2024 < best_rmse
        global best_rmse = rmse_2024
        global best_lambda = lam
    end
end

println("\n--- RÉSULTAT RIDGE ---")
println("Meilleur λ : ", best_lambda, " avec RMSE = ", best_rmse)
println("Rappel du simple GLM (Inertie) : 9.3808")


# 6. Entraînement Final & Création de la Soumission (Phase 4)

Entraînement du Modèle Ridge optimal (`λ = 1000.0`) sur **100% de l'historique** (2007-2024), traitement du `test` avec l'inertie continue et export Kaggle.

In [ ]:
using CSV, DataFrames

println("--- Diagnostic des données Test --- ")

# 1. Chargement des données test brutes
test_air = CSV.read("data/qualite-de-lair_test.csv", DataFrame)
test_meteo = CSV.read("data/meteo_test.csv", DataFrame)

# 2. Jointure sur la Date
test_df = leftjoin(test_air, test_meteo[:, 3:end], on=:Date)

println("\n--- Valeurs manquantes (missing) réelles ---")
for col in names(test_df)
    miss_count = sum(ismissing.(test_df[!, col]))
    if miss_count > 0
        println(" - ", col, " : ", miss_count, " manquants")
    end
end

println("\n--- Types des colonnes (Attention aux String cachés) ---")
for col in names(test_df)
    if eltype(test_df[!, col]) <: AbstractString || eltype(test_df[!, col]) >: AbstractString
        println(" ⚠️ ", col, " est lu comme du texte (String) !")
    end
end


In [ ]:
# ==========================================
# 3. Nettoyage Identique au Train Pipeline
# ==========================================

# --- Fonctions Imputation Temporelle ---
function forward_fill!(df, col)
    last_val = missing
    for i in 1:nrow(df)
        if !ismissing(df[i, col]) 
            last_val = df[i, col]
        else
            df[i, col] = last_val
        end
    end
end

function backward_fill!(df, col)
    next_val = missing
    for i in nrow(df):-1:1
        if !ismissing(df[i, col])
            next_val = df[i, col]
        else
            df[i, col] = next_val
        end
    end
end

# a) Suppression des colonnes interdites
cols_to_drop = [:neige, :neige_au_sol, :SO2]
intersect_cols = intersect(Symbol.(names(test_df)), cols_to_drop)
if !isempty(intersect_cols)
    select!(test_df, Not(intersect_cols))
end

# b) Imputation temporelle O3 et NO2 stricte (par bloc de station)
sort!(test_df, [:stationId, :Date])
for col in intersect([:O3, :NO2], Symbol.(names(test_df)))
    for g in groupby(test_df, :stationId)
        forward_fill!(g, col)
        backward_fill!(g, col)
    end
    
    # S'il en reste encore (station complètement vide l'hiver), on prend la médiane globale du Test
    med_fallback = median(skipmissing(model_df[!, col]))
    test_df[!, col] = coalesce.(test_df[!, col], med_fallback)
end

println("Nettoyage Train-like (Colonnes supprimées + Lags foward/backward) terminé sur le jeu test.")
println("Colonnes restantes : ", names(test_df))


In [ ]:
# ==========================================
# 4. Fabrication des Features et de l'Inertie (Phase Finale)
# ==========================================

println("Création des métadonnées et bases mathématiques...")

# Gère Missing + n'importe quel type numérique (Int64, Float32, etc.)
to_f64 = x -> ismissing(x) ? missing : Float64(x)

o3_center = median(skipmissing(to_f64.(model_df.O3)))  # ← to_f64 ici aussi

test_df.O3_centered_sq       = (to_f64.(test_df.O3) .- o3_center) .^ 2
test_df.inv_vitesse_vent_moy = 1.0 ./ (to_f64.(test_df.vitesse_vent_moy) .+ 1e-6)
test_df.inv_visibilite_moy   = 1.0 ./ (to_f64.(test_df.visibilite_moy)   .+ 1e-6)
test_df.hum_rel_moy_sq       = to_f64.(test_df.hum_rel_moy) .^ 2

println("Greffe temporelle avec Décembre 2024 pour permettre les Lags de Janvier 2025...")

# 3. L'Opération de chirurgie : Joindre la queue de 2024
train_tail = filter(row -> row.Date >= Date(2024, 12, 15), model_df) 

# On garde uniquement les colonnes qu'ils ont en commun
cols_intersect = intersect(names(train_tail), names(test_df))
test_df_clean = test_df[:, cols_intersect]
train_tail_clean = train_tail[:, cols_intersect]

test_fusion = vcat(train_tail_clean, test_df_clean, cols=:union)
sort!(test_fusion, [:stationId, :Date])

# 4. On lance l'Usine de la Feature Box !
println("Lancement de apply_feature_box! sur le pont temporel 2024-2025...")
apply_feature_box!(test_fusion, box_options)

# 5. Isolement Chirurgical
test_final = filter(row -> row.Date >= Date(2025, 1, 1), test_fusion)
println("Test_final est généré avec ", ncol(test_final), " colonnes réelles, prêt pour l'entraînement final.")


In [ ]:
# test_final a toutes les colonnes nécessaires ?
missing_cols = setdiff(string.(features_symb), names(test_final))
println("Colonnes manquantes dans test_final : ", missing_cols)

In [ ]:
# Quelles dates ont des missing dans les lags ?
for col in [:NO2_lag_1, :O3_lag_1, :vitesse_vent_moy_3j, :temp_moy_3j, :jours_sans_pluie]
    rows_miss = test_final[ismissing.(test_final[!, col]), [:Date, :stationId, col]]
    if nrow(rows_miss) > 0
        println("⚠ Missing dans ", col, " :")
        println(rows_miss)
    end
end

In [ ]:
last_o3 = last(filter(row -> isequal(row.stationId, 103) && !ismissing(row.O3), model_df).O3)
test_final[2191, :O3_lag_1] = Float64(last_o3)

println("✓ Imputé : ", last_o3)
println("Missing restants O3_lag_1 : ", sum(ismissing.(test_final.O3_lag_1)))

In [ ]:
# Vérification complète de tous les missing dans les features
println("Vérification des missing dans toutes les features...")
for col in features_symb
    n_miss = sum(ismissing.(test_final[!, col]))
    if n_miss > 0
        println("⚠ ", col, " : ", n_miss, " missing")
    end
end
println("✓ Vérification terminée")

In [ ]:
# ==========================================
# ENTRAÎNEMENT FINAL SUR TOUT LE DATASET
# ==========================================
println("Entraînement final sur toutes les données avec λ = 1000.0...")

λ_best = 1000.0

# 1. Extraction matrice complète
y_full, X_full = modelcols(f_schema, model_df_enriched)

# 2. Standardisation sur le dataset complet
μ_full = mean(X_full, dims=1)
σ_full = std(X_full, dims=1)
for j in 1:size(X_full, 2)
    if σ_full[1,j] == 0.0 || j == 1
        μ_full[1,j] = 0.0
        σ_full[1,j] = 1.0
    end
end

X_full_std = (X_full .- μ_full) ./ σ_full

# 3. Entraînement Ridge final
β_final = fit_ridge(X_full_std, Float64.(y_full), λ_best)
println("✓ Modèle entraîné — ", length(β_final), " coefficients")

# 4. RMSE sur le train complet (sanity check)
yhat_train = X_full_std * β_final
rmse_train_final = sqrt(mean((Float64.(y_full) .- yhat_train).^2))
println("RMSE sur train complet : ", round(rmse_train_final, digits=4))

# ==========================================
# PRÉDICTIONS SUR TEST_FINAL
# ==========================================
println("\nGénération des prédictions sur test_final...")

# 5. Vérification colonnes manquantes
missing_cols = setdiff(string.(features_symb), names(test_final))
if length(missing_cols) > 0
    println("⚠ Colonnes manquantes dans test_final : ", missing_cols)
else
    println("✓ Toutes les colonnes sont présentes")
end

# 6. Extraction X test
_, X_test = modelcols(f_schema.rhs, test_final)

# 7. Standardisation avec μ et σ du TRAIN (jamais recalculer sur test !)
X_test_std = (X_test .- μ_full) ./ σ_full

# 8. Prédictions
predictions = X_test_std * β_final

# 9. Sanity check
println("Prédictions générées : ", length(predictions), " lignes")
println("Min : ", round(minimum(predictions), digits=2), 
        " | Max : ", round(maximum(predictions), digits=2), 
        " | Moy : ", round(mean(predictions), digits=2))

# 10. Ajout au dataframe
test_final.PM_predicted = predictions

In [ ]:
println("Taille X_test_std : ", size(X_test_std))
println("Taille β_final : ", length(β_final))

In [ ]:
# 1. Copie propre pour ne pas corrompre test_final
test_for_pred = copy(test_final)
test_for_pred[!, :PM] = zeros(Float64, nrow(test_for_pred))

println("nrow test_for_pred : ", nrow(test_for_pred))

# 2. Extraction X
_, X_test = modelcols(f_schema, test_for_pred)
println("Taille X_test : ", size(X_test))

# 3. Standardisation
X_test_std = (X_test .- μ_full) ./ σ_full
println("Taille X_test_std : ", size(X_test_std))

# 4. Prédictions
predictions = X_test_std * β_final
println("Taille predictions : ", length(predictions))

# 5. Ajout à test_final
test_final[!, :PM_predicted] = predictions

println("Min : ", round(minimum(predictions), digits=2),
        " | Max : ", round(maximum(predictions), digits=2),
        " | Moy : ", round(mean(predictions), digits=2))

In [ ]:
using DataFrames, CSV, Dates

submission = DataFrame()

submission.id = string.(
    "(",
    "Date(\"", string.(test_final.Date), "\"), ",
    test_final.stationId,
    ")"
)

submission.PM = test_final.PM_predicted

CSV.write("submission.csv", submission)

first(submission, 5)

In [ ]:
println("Missing : ", count(ismissing, test_final.PM_predicted))
println("NaN : ", count(isnan, test_final.PM_predicted))
println("Inf : ", count(isinf, test_final.PM_predicted))

In [ ]:
println("Min : ", minimum(test_final.PM_predicted))
println("Max : ", maximum(test_final.PM_predicted))
println("Mean : ", mean(test_final.PM_predicted))

In [ ]:
using CSV, DataFrames, Statistics

# 1. Charger la submission
submission = CSV.read("submission.csv", DataFrame)

println("Colonnes : ", names(submission))
println("Taille : ", size(submission))
println()
println("Premières lignes :")
println(first(submission, 5))

# 2. Vérifier que PM existe
if !("PM" in names(submission))
    error("La colonne PM est absente du fichier submission.csv")
end

# 3. Vérifications de base
pm = submission.PM

println()
println("=== Vérifications PM ===")
println("Missing : ", count(ismissing, pm))
println("NaN : ", count(x -> x isa AbstractFloat && isnan(x), pm))
println("Inf : ", count(x -> x isa AbstractFloat && isinf(x), pm))
println("Valeurs uniques : ", length(unique(pm)))

# 4. Stats descriptives
println()
println("=== Stats PM ===")
println("Min : ", minimum(pm))
println("Max : ", maximum(pm))
println("Mean : ", mean(pm))
println("Median : ", median(pm))

qs = quantile(Float64.(pm), [0.5, 0.9, 0.95, 0.99])
println("Q50  : ", qs[1])
println("Q90  : ", qs[2])
println("Q95  : ", qs[3])
println("Q99  : ", qs[4])

# 5. Détection des pics
println()
println("=== Pics prédits ===")
println("Nb > 30  : ", count(>(30), pm))
println("Nb > 50  : ", count(>(50), pm))
println("Nb > 70  : ", count(>(70), pm))
println("Nb > 100 : ", count(>(100), pm))

# 6. Vérifier les valeurs négatives
println()
println("Nb < 0 : ", count(<(0), pm))

# 7. Vérifier format id si présent
if "ID" in names(submission)
    println()
    println("=== Vérification ID ===")
    println("Nb ID uniques : ", length(unique(submission.ID)))
    println("Exemples ID : ")
    println(first(submission.ID, 5))
end